# 🏠 Actividad integradora — Unidades 4 y 5
### Python para Ciencia de Datos | UADE
#### Dataset: Airbnb Buenos Aires (alojamientos + hosts)

---

En esta actividad integramos todo lo visto en las unidades 4 y 5:

| Parte | Tema | Herramientas |
|---|---|---|
| 0 | Carga, limpieza y **left join** | `pd.read_csv`, `pd.merge(how="left")` |
| A | **Estadística descriptiva** sobre `hosts` | media, mediana, moda, desvío, cuartiles, RIC, histograma, boxplot |
| B | **Análisis bivariado** sobre el dataset combinado | scatterplot, boxplot por categoría, pairplot, `corr()`, heatmap |
| C | **Agregaciones** para replicar gráficos de [Inside Airbnb — Buenos Aires](https://insideairbnb.com/es/buenos-aires/) | `groupby`, `agg`, `value_counts`, `pd.cut` |

> 💡 En la U4 ya hicimos la estadística descriptiva de **alojamientos**, así que ahora el foco univariado está puesto en los **hosts**.

En cada sección vas a encontrar celdas **✍️ Para responder**: escribí tu interpretación con tus palabras. Un gráfico sin conclusión no es un análisis.

# 1. Importar librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Settings generales para visualizar
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

# 2. Cargar datasets

In [ ]:
# Load dataset airbnb_jr.csv con datos de alojamientos
alojamientos = pd.read_csv("https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_jr.csv")
print(f"Dataset airbnb_jr loaded {alojamientos.shape}")

# Load dataset airbnb_hosts con datos de los hosts
hosts = pd.read_csv("https://raw.githubusercontent.com/UADE-Python-Data-Science/583433_repo_oficial/refs/heads/main/Datasets/airbnb_hosts.csv")
print(f"Dataset airbnb_hosts loaded {hosts.shape}")

# 3. Data quality & transformation

## 3.1 Alojamientos

In [ ]:
# normalizar columnas
alojamientos.columns = (
    alojamientos.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
)

# drop duplicados
alojamientos.drop_duplicates(inplace=True)

# renombrar columna neighbourhood a barrio
alojamientos.rename(
    columns={
        "neighbourhood": "barrio",
    },
    inplace=True,
)

# drop NaN
alojamientos.dropna(inplace=True)

# transformar price de object a float
try:
    alojamientos["price"] = alojamientos["price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False)
    alojamientos["price"] = pd.to_numeric(alojamientos["price"], errors="coerce")
    alojamientos["price_usd"] = round(alojamientos["price"] / 1500, 2)
except Exception as e:
    print(f"No se pudo transformar price {e}")

# transformar last_review a datetime
try:
    alojamientos["last_review"] = pd.to_datetime(alojamientos["last_review"], format="%Y-%m-%d", errors="coerce")
except Exception as e:
    print(f"No se pudo transformar last_review {e}")

print(f"alojamientos limpio: {alojamientos.shape}")

## 3.2 Hosts

In [ ]:
# cantidad de registros duplicados
print(f"Registros duplicados en hosts: {hosts.duplicated().sum()}")

# visualizar registros duplicados
# hosts[hosts.duplicated(keep=False)].sort_values(by="host_id")

# drop registros duplicados
hosts.drop_duplicates(inplace=True)

# validamos que ahora host_id sea una clave única (1 fila por host)
print(f"host_id repetidos luego de limpiar: {hosts['host_id'].duplicated().sum()}")
print(f"hosts limpio: {hosts.shape}")

In [ ]:
hosts.info()

# 4. Left Join alojamientos - hosts

Usamos **left join** porque la unidad de análisis es el **alojamiento**: queremos conservar todos los alojamientos, tengan o no perfil de host.

> 💡 `validate="m:1"` le pide a pandas que verifique que la relación sea **muchos alojamientos → 1 host**. Si `hosts` tuviera `host_id` repetidos, el merge fallaría en vez de duplicar filas silenciosamente.

In [ ]:
merge = pd.merge(alojamientos, hosts, on="host_id", how="left", validate="m:1")

print(f"alojamientos: {alojamientos.shape[0]} filas")
print(f"LEFT JOIN:    {merge.shape[0]} filas  <- debe coincidir con alojamientos")

In [ ]:
merge.info()

In [ ]:
# ¿Cuántos alojamientos tiene cada host? (dentro de nuestra muestra)
merge.groupby("host_id").size().sort_values(ascending=False).head()

---
# Parte A — Estadística descriptiva sobre `hosts`

Trabajamos sobre el dataset **`hosts`** completo (no sobre el merge), porque la unidad de observación ahora es **el anfitrión**.

| Variable | Tipo | Qué vamos a mirar |
|---|---|---|
| `hosts_time_as_user_years` | cuantitativa discreta | posición central, dispersión, histograma, boxplot |
| `host_identity_verified` | cualitativa | frecuencias y moda |
| `host_location` | cualitativa | moda, top ubicaciones |
| `host_about` | texto | ¿completó su descripción? → la convertimos en variable cualitativa |

## A.1 Feature engineering sobre hosts

Antes de describir, creamos algunas variables más fáciles de analizar:

* `verificado`: `host_identity_verified` pasa de `"t"/"f"` a booleano.
* `tiene_about`: ¿el host escribió una descripción? (`host_about` no nulo y distinto de `"-"`).
* `es_local`: ¿el host declara vivir en Buenos Aires? (`host_location` contiene `"Buenos Aires"`).

In [ ]:
hosts["verificado"] = hosts["host_identity_verified"] == "t"
hosts["tiene_about"] = hosts["host_about"].notna() & (hosts["host_about"].str.strip() != "-")
hosts["es_local"] = hosts["host_location"].str.contains("Buenos Aires", case=False, na=False)

hosts[["host_name", "host_location", "verificado", "tiene_about", "es_local", "hosts_time_as_user_years"]].head()

## A.2 Variables cualitativas: frecuencias y moda

In [ ]:
# Proporción (%) de hosts en cada categoría
for col in ["verificado", "tiene_about", "es_local"]:
    print(f"--- {col} ---")
    print((hosts[col].value_counts(normalize=True) * 100).round(1).to_string())
    print()

In [ ]:
# Moda y top 10 de ubicaciones declaradas
print(f"Moda de host_location: {hosts['host_location'].mode()[0]}")
print(f"Hosts sin ubicación declarada: {hosts['host_location'].isna().sum()}")
hosts["host_location"].value_counts().head(10)

✍️ **Para responder:**
1. ¿Qué porcentaje de hosts tiene la identidad verificada? ¿Te sirve esta variable para diferenciar hosts?
2. ¿Qué proporción de hosts declara vivir fuera de Buenos Aires? ¿Qué ubicaciones extranjeras aparecen?

## A.3 Medidas de tendencia central — `hosts_time_as_user_years`

In [ ]:
antiguedad = hosts["hosts_time_as_user_years"]

media   = antiguedad.mean()
mediana = antiguedad.median()
moda    = antiguedad.mode()[0]

print(f"Media:   {media:.2f} años")
print(f"Mediana: {mediana:.2f} años")
print(f"Moda:    {moda} años")

## A.4 Medidas de dispersión y cuartiles

In [ ]:
desvio = antiguedad.std()
rango  = antiguedad.max() - antiguedad.min()
Q1 = antiguedad.quantile(0.25)
Q3 = antiguedad.quantile(0.75)
RIC = Q3 - Q1
limite_inf = Q1 - 1.5 * RIC
limite_sup = Q3 + 1.5 * RIC

print(f"Desvío estándar: {desvio:.2f} años")
print(f"Rango:           {rango} años  ({antiguedad.min()} → {antiguedad.max()})")
print(f"Q1: {Q1}  |  Q3: {Q3}  |  RIC: {RIC}")
print(f"Límites de Tukey: [{limite_inf}, {limite_sup}]")
print(f"Outliers: {((antiguedad < limite_inf) | (antiguedad > limite_sup)).sum()}")

In [ ]:
antiguedad.describe()

## A.5 Histograma y boxplot

Reutilizamos la función `plot_histograma_full` de la U4.

In [ ]:
def plot_histograma_full(df, variable, desvio=False, bins="auto"):

    sns.histplot(data=df, x=variable, bins=bins, kde=True)

    media = df[variable].mean()
    mediana = df[variable].median()
    std = df[variable].std()

    plt.axvline(media, linestyle="-", color="red", label=f"Media: {media:.2f}")
    plt.axvline(mediana, linestyle="--", color="orange", label=f"Mediana: {mediana:.2f}")

    if desvio:
        plt.axvline(media + std, linestyle="--", color="green", label=f"+1std: {media + std:.2f}")
        plt.axvline(media - std, linestyle="--", color="green", label=f"-1std: {media - std:.2f}")

    plt.title(f"Distribución de {variable}")
    plt.xlabel(variable)
    plt.ylabel("Frecuencia")
    plt.legend()

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
# bins=range(0, 19) -> un bin por cada año posible (0 a 17)
plot_histograma_full(hosts, "hosts_time_as_user_years", desvio=True, bins=range(0, 19))

plt.subplot(1, 2, 2)
sns.boxplot(data=hosts, x="hosts_time_as_user_years")
plt.title("Boxplot de hosts_time_as_user_years")

plt.tight_layout()
plt.show()

✍️ **Para responder:**
1. ¿La distribución es simétrica? ¿Por qué la media es **menor** que la mediana?
2. ¿Ves más de un "pico" en el histograma? ¿Qué podría explicar un grupo grande de hosts muy nuevos (0–3 años)?
3. ¿Hay outliers según el criterio de Tukey?

## A.6 ¿Cambia la antigüedad según el perfil del host?

Combinamos una variable cualitativa con una cuantitativa: `groupby` + `describe()` y boxplots por grupo.

In [ ]:
hosts.groupby("es_local")["hosts_time_as_user_years"].describe()

In [ ]:
hosts.columns

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
sns.boxplot(data=hosts, x="hosts_time_as_user_years", hue="es_local", orient="h")
plt.title("Antigüedad según si el host es local")

plt.subplot(1, 2, 2)
sns.boxplot(data=hosts, x="hosts_time_as_user_years", hue="tiene_about", orient="h")
plt.title("Antigüedad según si completó su descripción")

plt.tight_layout()
plt.show()

✍️ **Para responder:** ¿Los hosts que completaron su descripción son más antiguos? ¿Y los locales vs. los que viven afuera?

---
# Parte B — Análisis bivariado sobre el dataset combinado (`merge`)

Ahora la unidad de observación vuelve a ser **el alojamiento**, pero enriquecido con los datos de su host.

Preguntas que vamos a responder:
1. ¿Los hosts con más antigüedad tienen mejores `review_scores`?
2. ¿El precio aumenta con la cantidad de `bedrooms`?
3. ¿Los alojamientos con más reseñas tienen mejor puntaje?
4. ¿Qué variables numéricas están más correlacionadas entre sí?

## B.1 Preparar el dataset: outliers de precio

Antes de relacionar variables, revisamos `price_usd`. Un solo precio extremo puede "aplastar" un scatterplot y distorsionar la correlación de Pearson (que es sensible a outliers).

In [ ]:
merge["price_usd"].describe()

In [ ]:
# Criterio de Tukey sobre price_usd (U4)
Q1 = merge["price_usd"].quantile(0.25)
Q3 = merge["price_usd"].quantile(0.75)
RIC = Q3 - Q1
limite_sup = Q3 + 1.5 * RIC

df = merge[merge["price_usd"] <= limite_sup].copy()

print(f"Límite superior: USD {limite_sup:.2f}")
print(f"Alojamientos descartados por precio atípico: {merge.shape[0] - df.shape[0]}")
print(f"Dataset para el análisis bivariado: {df.shape}")

In [ ]:
# Traemos también las features de hosts creadas en la Parte A
df["es_local"] = df["host_location"].str.contains("Buenos Aires", case=False, na=False)
df["tiene_about"] = df["host_about"].notna() & (df["host_about"].str.strip() != "-")

## B.2 `review_scores` vs `hosts_time_as_user_years`

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
sns.scatterplot(data=df, x="hosts_time_as_user_years", y="review_scores", alpha=0.5)
plt.title("Puntaje vs antigüedad del host")

plt.subplot(1, 2, 2)
# Como la antigüedad es discreta, un boxplot por año se lee mejor que el scatter
sns.boxplot(data=df, x="hosts_time_as_user_years", y="review_scores", color="steelblue")
plt.title("Puntaje por año de antigüedad del host")

plt.tight_layout()
plt.show()

print(f"Correlación de Pearson: {df['hosts_time_as_user_years'].corr(df['review_scores']):.3f}")

✍️ **Para responder:** Según la escala vista en la U4 (0–0.25 sin relación, 0.25–0.5 débil…), ¿qué tipo de relación hay? ¿Por qué casi todos los puntos están arriba de 4.5? (pista: mirá `df["review_scores"].describe()`).

## B.3 `price_usd` vs `bedrooms`

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
sns.scatterplot(data=df, x="bedrooms", y="price_usd", alpha=0.5)
plt.title("Precio vs dormitorios")

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x="bedrooms", y="price_usd", color="steelblue")
plt.title("Distribución del precio por cantidad de dormitorios")

plt.tight_layout()
plt.show()

print(f"Correlación de Pearson: {df['bedrooms'].corr(df['price_usd']):.3f}")

In [ ]:
# Complementamos con una agregación: precio mediano y cantidad de alojamientos por bedrooms
df.groupby("bedrooms").agg(
    cantidad=("price_usd", "count"),
    precio_mediano=("price_usd", "median"),
    precio_promedio=("price_usd", "mean"),
)

✍️ **Para responder:** ¿El precio crece con los dormitorios? ¿Qué tan confiables son las conclusiones para 4 o más dormitorios, viendo la columna `cantidad`?

## B.4 `review_scores` vs `number_reviews`

In [ ]:
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df, x="number_reviews", y="review_scores", hue="es_local", alpha=0.6)
plt.title("Puntaje vs cantidad de reseñas")
plt.show()

print(f"Correlación de Pearson: {df['number_reviews'].corr(df['review_scores']):.3f}")

✍️ **Para responder:** ¿Qué pasa con la **dispersión** del puntaje a medida que aumentan las reseñas? ¿Por qué un alojamiento con 1 o 2 reseñas puede tener puntajes extremos?

## B.5 Variable cualitativa vs cuantitativa: precio según perfil del host

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 2, 1)
sns.boxplot(data=df, x="price_usd", y="es_local", orient="h")
plt.title("Precio según si el host es local")

plt.subplot(1, 2, 2)
sns.kdeplot(data=df, x="price_usd", hue="tiene_about", fill=True, common_norm=False)
plt.title("Densidad del precio según si el host tiene descripción")

plt.tight_layout()
plt.show()

df.groupby("es_local")["price_usd"].describe()

## B.6 Pairplot

In [ ]:
v_numericas = ["price_usd", "bedrooms", "number_reviews", "review_scores", "hosts_time_as_user_years"]
sns.pairplot(data=df, vars=v_numericas, plot_kws={"alpha": 0.4})
plt.show()

## B.7 Matriz de correlación y mapa de calor

In [ ]:
corr_matrix = df[v_numericas].corr()
corr_matrix

In [ ]:
plt.figure(figsize=(8, 6))
# Paleta divergente centrada en 0: azul = correlación positiva, rojo = negativa
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, center=0)
plt.title("Matriz de correlación — alojamientos + hosts")
plt.show()

✍️ **Para responder:**
1. ¿Cuál es el par de variables con la correlación más fuerte? Clasificala según la escala de la U4.
2. ¿Encontraste alguna relación entre variables de **alojamientos** y variables de **hosts**? ¿Qué nos dice eso sobre el valor del merge?
3. Correlación ≠ causalidad: elegí un par correlacionado y proponé una variable que no está en el dataset y que podría explicar ambas.

---
# Parte C — Agregaciones: replicando Inside Airbnb

El sitio [Inside Airbnb — Buenos Aires](https://insideairbnb.com/es/buenos-aires/) publica un tablero con indicadores sobre los alojamientos de la ciudad. Vamos a replicar sus gráficos principales con `groupby` / `agg` sobre **nuestra muestra**.

| Gráfico de Inside Airbnb | Qué agregamos |
|---|---|
| **Tipo de habitación** | % de alojamientos por `room_type` y precio promedio |
| **Actividad** | alojamientos con reseña en los últimos 12 meses |
| **Anuncios por anfitrión** | hosts con 1 alojamiento vs. multi-anfitriones |
| **Principales anfitriones** | ranking de hosts por cantidad de alojamientos |
| **Barrios** (mapa) | cantidad y precio por `barrio` + mapa con latitud/longitud |

> ⚠️ Nuestra muestra tiene ~700 alojamientos (Inside Airbnb trabaja con decenas de miles), así que los números van a diferir. Lo importante es **replicar el método**.

Usamos `merge` (con todos los alojamientos) para conteos, y `df` (sin outliers) para precios.

## C.1 Tipo de habitación

In [ ]:
room = (
    merge.groupby("room_type", as_index=False)
    .agg(
        alojamientos=("host_id", "count"),
        precio_mediano_usd=("price_usd", "median"),  # mediana: merge todavía tiene outliers de precio
    )
    .sort_values("alojamientos", ascending=False)
)
room["porcentaje"] = (room["alojamientos"] / room["alojamientos"].sum() * 100).round(1)
room

In [ ]:
plt.figure(figsize=(10, 3))
ax = sns.barplot(data=room, x="porcentaje", y="room_type", color="steelblue")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=4)
plt.xlabel("% de alojamientos")
plt.ylabel("")
plt.xlim(0, 110)
plt.title("Tipo de habitación — Airbnb Buenos Aires")
plt.show()

> 💡 Inside Airbnb destaca el % de **alojamientos enteros** porque son los que más compiten con el alquiler tradicional de viviendas.

## C.2 Actividad: ¿cuántos alojamientos están activos?

Inside Airbnb considera **activo** a un alojamiento con al menos una reseña en los últimos 12 meses. Como referencia de "hoy" tomamos la fecha de reseña más reciente del dataset.

In [ ]:
fecha_ref = merge["last_review"].max()
merge["activo"] = merge["last_review"] >= fecha_ref - pd.DateOffset(months=12)
merge["anio_last_review"] = merge["last_review"].dt.year

print(f"Fecha de referencia: {fecha_ref.date()}")
print(f"Alojamientos activos (últimos 12 meses): {merge['activo'].sum()} de {len(merge)} "
      f"({merge['activo'].mean() * 100:.1f}%)")

In [ ]:
actividad = merge.groupby("activo").agg(
    alojamientos=("host_id", "count"),
    reviews_promedio=("number_reviews", "mean"),
    puntaje_promedio=("review_scores", "mean"),
    precio_mediano_usd=("price_usd", "median"),
)
actividad

In [ ]:
por_anio = merge.groupby("anio_last_review", as_index=False).agg(alojamientos=("host_id", "count"))

plt.figure(figsize=(10, 4))
sns.barplot(data=por_anio, x="anio_last_review", y="alojamientos", color="steelblue")
plt.xlabel("Año de la última reseña")
plt.ylabel("Cantidad de alojamientos")
plt.title("Alojamientos según año de su última reseña")
plt.show()

## C.3 Anuncios por anfitrión: ¿un solo alojamiento o multi-anfitrión?

Primero **agregamos por host** para saber cuántos alojamientos tiene cada uno, y después **clasificamos** con `pd.cut` (convierte una variable numérica en categorías por rangos).

In [ ]:
por_host = merge.groupby(["host_id", "host_name"], as_index=False).agg(
    alojamientos=("barrio", "count"),
    precio_promedio_usd=("price_usd", "mean"),
    antiguedad=("hosts_time_as_user_years", "first"),
)

por_host["categoria"] = pd.cut(
    por_host["alojamientos"],
    bins=[0, 1, 100],
    labels=["1 alojamiento", "2 o más alojamientos"],
)
por_host.sort_values("alojamientos", ascending=False).head()

In [ ]:
resumen_hosts = por_host.groupby("categoria", observed=True).agg(
    hosts=("host_id", "count"),
    alojamientos=("alojamientos", "sum"),
)
resumen_hosts["% hosts"] = (resumen_hosts["hosts"] / resumen_hosts["hosts"].sum() * 100).round(1)
resumen_hosts["% alojamientos"] = (resumen_hosts["alojamientos"] / resumen_hosts["alojamientos"].sum() * 100).round(1)
resumen_hosts

In [ ]:
# Transformamos a formato largo (como en U5_NB1) para comparar ambos porcentajes
resumen_long = resumen_hosts.reset_index().melt(
    id_vars="categoria",
    value_vars=["% hosts", "% alojamientos"],
    var_name="métrica",
    value_name="porcentaje",
)

plt.figure(figsize=(10, 4))
ax = sns.barplot(data=resumen_long, x="porcentaje", y="categoria", hue="métrica")
for cont in ax.containers:
    ax.bar_label(cont, fmt="%.1f%%", padding=4)
plt.xlim(0, 110)
plt.xlabel("%")
plt.ylabel("")
plt.title("Anuncios por anfitrión — Airbnb Buenos Aires")
plt.show()

✍️ **Para responder:** ¿Por qué el % de alojamientos de los multi-anfitriones es **mayor** que su % de hosts? ¿Por qué a Inside Airbnb le interesa este indicador?

## C.4 Principales anfitriones

In [ ]:
top_hosts = (
    merge.groupby(["host_id", "host_name"], as_index=False)
    .agg(
        alojamientos=("barrio", "count"),
        barrios=("barrio", "nunique"),
        precio_promedio_usd=("price_usd", "mean"),
        puntaje_promedio=("review_scores", "mean"),
        antiguedad=("hosts_time_as_user_years", "first"),
    )
    .sort_values(["alojamientos", "puntaje_promedio"], ascending=False)
    .head(10)
    .reset_index(drop=True)
)
top_hosts

✍️ **Para responder:** esta tabla usa `merge` (con outliers). ¿Algún host tiene un precio promedio sospechoso? Buscá sus alojamientos con un filtro y decidí: ¿error de carga o alojamiento de lujo? ¿Qué medida usarías en su lugar?

## C.5 Barrios: cantidad y precio

In [ ]:
barrios = (
    df.groupby("barrio", as_index=False)
    .agg(
        alojamientos=("price_usd", "count"),
        precio_mediano_usd=("price_usd", "median"),
        puntaje_promedio=("review_scores", "mean"),
        pct_hosts_locales=("es_local", "mean"),
    )
    .sort_values("alojamientos", ascending=False)
)
barrios["pct_hosts_locales"] = (barrios["pct_hosts_locales"] * 100).round(1)
barrios.head(10)

In [ ]:
# Solo barrios con al menos 10 alojamientos: con menos, el promedio no es representativo
barrios_top = barrios[barrios["alojamientos"] >= 10]

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.barplot(data=barrios_top, x="alojamientos", y="barrio", color="steelblue")
plt.title("Alojamientos por barrio (≥ 10)")
plt.ylabel("")

plt.subplot(1, 2, 2)
sns.barplot(
    data=barrios_top.sort_values("precio_mediano_usd", ascending=False),
    x="precio_mediano_usd", y="barrio", color="steelblue",
)
plt.title("Precio mediano por barrio (USD)")
plt.ylabel("")

plt.tight_layout()
plt.show()

## C.6 El mapa: un scatterplot con latitud y longitud

El gráfico principal de Inside Airbnb es un **mapa**. ¡Un mapa de puntos no es más que un **scatterplot** donde `x = longitude` e `y = latitude`!

In [ ]:
plt.figure(figsize=(9, 8))
sns.scatterplot(
    data=df, x="longitude", y="latitude",
    hue="price_usd", palette="viridis", size="bedrooms", sizes=(15, 120), alpha=0.7,
)
plt.title("Mapa de alojamientos — color = precio (USD), tamaño = dormitorios")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.gca().set_aspect("equal")
plt.show()

✍️ **Para responder:** ¿En qué zona de la ciudad se concentran los alojamientos? ¿Y los más caros? Relacionalo con la tabla de barrios de C.5.

---
# 🔑 Conclusiones finales

Escribí un párrafo (5–8 líneas) respondiendo:

1. ¿Cómo es el **perfil típico de un host** de Airbnb en Buenos Aires? (Parte A)
2. ¿Qué variables explican mejor el **precio** y el **puntaje** de un alojamiento? ¿Aportaron algo las variables del host? (Parte B)
3. ¿Qué indicador de Inside Airbnb te pareció más revelador y por qué? (Parte C)

| Tema | Dónde lo usamos |
|---|---|
| `drop_duplicates`, `dropna`, conversión de tipos | Sección 3 |
| `pd.merge(how="left", validate="m:1")` | Sección 4 |
| media, mediana, moda, desvío, cuartiles, Tukey | Parte A |
| histograma, boxplot, kdeplot | Partes A y B |
| scatterplot, pairplot, `corr()`, heatmap | Parte B |
| `groupby`, `agg`, `value_counts`, `pd.cut`, `melt` | Parte C |